In [2]:
from hybrid_automaton import Automaton, State, Transition
import numpy as np
import time

In [3]:
TARGET_SPEED = 30.0  # m/s (108 km/h)
SAFE_DISTANCE = 50.0  # meters
CAR_AHEAD_SPEED = 20.0  # m/s

In [4]:
def accelerate_flow(x, aux_x, u, ctx, dt):
    """Accelerate at 2 m/s² - x = [v, dist]"""
    v_dot = 2.0 if x[0] < TARGET_SPEED else 0.0
    dist_dot = -(x[0] - CAR_AHEAD_SPEED)
    return np.array([v_dot, dist_dot])

def cruise_flow(x, aux_x, u, ctx, dt):
    """Maintain constant speed"""
    return np.array([0.0, -(x[0] - CAR_AHEAD_SPEED)])

def brake_flow(x, aux_x, u, ctx, dt):
    """Gentle braking at -1.5 m/s²"""
    v_dot = -1.5 if x[0] > 0 else 0.0
    return np.array([v_dot, -(x[0] - CAR_AHEAD_SPEED)])

def emergency_brake_flow(x, aux_x, u, ctx, dt):
    """Hard braking at -5 m/s²"""
    v_dot = -5.0 if x[0] > 0 else 0.0
    return np.array([v_dot, -(x[0] - CAR_AHEAD_SPEED)])

In [5]:
# Guards - x = [v, dist]
def reached_target_speed(x, aux_x, u, ctx, dt):
    return abs(x[0] - TARGET_SPEED) < 0.5

def below_target_speed(x, aux_x, u, ctx, dt):
    return x[0] < TARGET_SPEED - 1.0 and x[1] > SAFE_DISTANCE

def too_close(x, aux_x, u, ctx, dt):
    return x[1] < SAFE_DISTANCE

def dangerously_close(x, aux_x, u, ctx, dt):
    return x[1] < 20.0

def safe_distance_restored(x, aux_x, u, ctx, dt):
    return x[1] > SAFE_DISTANCE + 10.0

In [6]:
# Callbacks
def on_accelerate():
    print("🚗💨 ACCELERATE mode")

def on_cruise():
    print("🚗➡️  CRUISE mode - Maintaining speed")

def on_brake():
    print("🚗🟡 BRAKE mode - Slowing down")

def on_emergency():
    print("🚗🔴 EMERGENCY BRAKE!")

In [7]:
accelerate = State("ACCELERATE", initial=True, flow=accelerate_flow, on_enter=on_accelerate)
cruise = State("CRUISE", flow=cruise_flow, on_enter=on_cruise)
brake = State("BRAKE", flow=brake_flow, on_enter=on_brake)
emergency = State("EMERGENCY_BRAKE", flow=emergency_brake_flow, on_enter=on_emergency)

In [8]:
# Transitions
accelerate.add_transition(Transition("reach_cruise", cruise, guards=[reached_target_speed], priority=1))
accelerate.add_transition(Transition("acc_to_brake", brake, guards=[too_close], priority=2))
accelerate.add_transition(Transition("acc_to_emergency", emergency, guards=[dangerously_close], priority=3))

cruise.add_transition(Transition("cruise_to_brake", brake, guards=[too_close], priority=1))
cruise.add_transition(Transition("cruise_to_emergency", emergency, guards=[dangerously_close], priority=2))
cruise.add_transition(Transition("cruise_to_acc", accelerate, guards=[below_target_speed], priority=3))

brake.add_transition(Transition("brake_to_emergency", emergency, guards=[dangerously_close], priority=1))
brake.add_transition(Transition("brake_to_cruise", cruise, guards=[safe_distance_restored], priority=2))

emergency.add_transition(Transition("emergency_to_brake", brake, guards=[safe_distance_restored], priority=1))

In [9]:
# Create automaton
car = Automaton(name="Cruise Control", states=[accelerate, cruise, brake, emergency], dt=0.1)

# Initial: x = [velocity, distance_to_car_ahead]
x0 = np.array([20.0, 100.0])
car.activate(x0=x0)

In [10]:
print("Starting cruise control simulation...")
print(f"Target speed: {TARGET_SPEED} m/s")
print()

# Simulate
for i in range(300):
    result = car.step()
    if i % 30 == 0:
        print(f"t={i*0.1:.1f}s | {car.q.name:15s} | Speed: {car.x[0]:5.1f} m/s | Distance: {car.x[1]:5.1f}m")
    if result and result.transition_taken:
        print(f"  ⚡ Transition: {result.transition_taken.name}")

print()

Starting cruise control simulation...
Target speed: 30.0 m/s

t=0.0s | ACCELERATE      | Speed:  20.2 m/s | Distance: 100.0m
t=3.0s | ACCELERATE      | Speed:  26.2 m/s | Distance:  90.7m
🚗➡️  CRUISE mode - Maintaining speed
  ⚡ Transition: reach_cruise
t=6.0s | CRUISE          | Speed:  29.6 m/s | Distance:  65.0m
🚗🟡 BRAKE mode - Slowing down
  ⚡ Transition: cruise_to_brake
t=9.0s | BRAKE           | Speed:  27.5 m/s | Distance:  37.5m
t=12.0s | BRAKE           | Speed:  23.0 m/s | Distance:  21.6m
🚗🔴 EMERGENCY BRAKE!
  ⚡ Transition: brake_to_emergency
t=15.0s | EMERGENCY_BRAKE | Speed:  10.1 m/s | Distance:  28.7m
🚗🟡 BRAKE mode - Slowing down
  ⚡ Transition: emergency_to_brake
🚗➡️  CRUISE mode - Maintaining speed
  ⚡ Transition: brake_to_cruise
🚗💨 ACCELERATE mode
  ⚡ Transition: cruise_to_acc
t=18.0s | ACCELERATE      | Speed:   1.0 m/s | Distance:  78.0m
t=21.0s | ACCELERATE      | Speed:   7.0 m/s | Distance: 126.3m
t=24.0s | ACCELERATE      | Speed:  13.0 m/s | Distance: 156.6m
t=

```yaml
information:
    hybrid_automaton: v0.0.1
    python: 3.12
```